# Логистическая регрессия с нуля

Реализация через градиентный спуск с функцией потерь Binary Cross-Entropy.  
Верификация: сравнение accuracy с `sklearn.linear_model.LogisticRegression` на датасете breast_cancer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score,
    f1_score, roc_auc_score
)

In [ ]:
class LogisticRegressionGD:
    """
    Логистическая регрессия через градиентный спуск (Binary Cross-Entropy).
    """

    def __init__(self, lr=0.01, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter
        self.w_ = None
        self.loss_history_ = []

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n = X.shape[0]

        # Добавляем столбец единиц для bias
        X_b = np.hstack([X, np.ones((n, 1))])

        # Инициализируем веса нулями
        self.w_ = np.zeros(X_b.shape[1])
        self.loss_history_ = []

        for _ in range(self.n_iter):
            # Предсказание вероятностей
            y_pred = self.predict_proba(X)

            # Градиент BCE: X^T (ŷ - y) / n
            grad = (1 / n) * (X_b.T @ (y_pred - y))

            # Обновить веса
            self.w_ = self.w_ - self.lr * grad

            # Сохранить loss
            loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
            self.loss_history_.append(loss)

        return self

    def predict_proba(self, X):
        X_b = np.hstack([X, np.ones((X.shape[0], 1))])
        return self.sigmoid(X_b @ self.w_)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
# Загружаем данные и масштабируем
X, y = load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X)

# Наша модель
model = LogisticRegressionGD(lr=0.1, n_iter=1000)
model.fit(X, y)

# sklearn для сравнения
sk_model = LogisticRegression(max_iter=1000)
sk_model.fit(X, y)

print('Наша accuracy:   ', accuracy_score(y, model.predict(X)))
print('sklearn accuracy:', accuracy_score(y, sk_model.predict(X)))

In [ ]:
# График сходимости
plt.plot(model.loss_history_)
plt.xlabel('Итерация')
plt.ylabel('BCE Loss')
plt.title('Сходимость градиентного спуска')
plt.show()

In [ ]:
# Метрики
y_pred = model.predict(X)
y_proba = model.predict_proba(X)

print('Confusion matrix:\n', confusion_matrix(y, y_pred))
print('Accuracy: ', accuracy_score(y, y_pred))
print('Precision:', precision_score(y, y_pred))
print('Recall:   ', recall_score(y, y_pred))
print('F1:       ', f1_score(y, y_pred))
print('ROC-AUC:  ', roc_auc_score(y, y_proba))